# Delta Lake no AWS Glue 6.0

Neste exemplo, que roda no runtime **AWS Glue 6.0**, seguimos a mesma sequência em **PySpark** e em **Spark SQL**: gravamos um CSV na `landing`, lemos do `s3a`, criamos uma tabela **externa** na camada `bronze` e atualizamos com `MERGE INTO`. Todas as células podem ser reexecutadas.

## 1. Sessão

O runtime **AWS Glue 6.0** já configura extensões, catálogos e acesso ao `s3a`; só precisamos obter a sessão a partir do `GlueContext`.

In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext

glueContext = GlueContext(SparkContext.getOrCreate())
spark = glueContext.spark_session

## 2. Preparar a origem

Gravamos duas levas de CSV (sem cabeçalho) em `landing/`: a carga inicial e as alterações. `overwrite` mantém a célula reexecutável.

In [ ]:
schema = "id INT, nome STRING, area STRING, ts BIGINT"

carga_inicial = spark.createDataFrame(
    [(1, "Ana", "Dados", 1), (2, "Bruno", "Plataforma", 1), (3, "Carla", "Segurança", 1)],
    schema,
)
alteracoes = spark.createDataFrame(
    [(2, "Bruno", "Dados", 2), (4, "Diego", "Plataforma", 2)],
    schema,
)

carga_inicial.write.mode("overwrite").csv("s3a://notislake/landing/glue_delta_employees/")
alteracoes.write.mode("overwrite").csv("s3a://notislake/landing/glue_delta_employees_updates/")

## 3. Ler do s3a

Em PySpark informamos o schema ao ler o CSV.

In [ ]:
origem = spark.read.schema(schema).csv("s3a://notislake/landing/glue_delta_employees/")
echo(origem.orderBy("id"), n=20)

Em Spark SQL lemos o mesmo caminho com `csv.` e convertemos as colunas (`_c0`, `_c1`, ...).

In [ ]:
SELECT
    CAST(_c0 AS INT) AS id,
    _c1 AS nome,
    _c2 AS area,
    CAST(_c3 AS BIGINT) AS ts
FROM csv.`s3a://notislake/landing/glue_delta_employees/`
ORDER BY id

## 4. Criar a tabela externa

Criamos a tabela Delta Lake no Catalog com `LOCATION` na camada `bronze`. `IF NOT EXISTS` evita erro ao reexecutar.

In [ ]:
CREATE SCHEMA IF NOT EXISTS bronze

In [ ]:
CREATE TABLE IF NOT EXISTS bronze.glue_delta_employees (
    id INT,
    nome STRING,
    area STRING,
    ts BIGINT
)
USING delta
LOCATION 's3a://notislake/bronze/glue_delta_employees_files'

## 5. Carregar e atualizar com MERGE INTO

O `MERGE INTO` atualiza as linhas com o mesmo `id` e insere as novas, por isso reexecutar não duplica dados. Primeiro, em SQL, aplicamos a carga inicial a partir de uma view temporária lida do `s3a`.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW carga_inicial_v AS
SELECT
    CAST(_c0 AS INT) AS id,
    _c1 AS nome,
    _c2 AS area,
    CAST(_c3 AS BIGINT) AS ts
FROM csv.`s3a://notislake/landing/glue_delta_employees/`

In [ ]:
MERGE INTO bronze.glue_delta_employees AS t
USING carga_inicial_v AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

Agora, em PySpark, lemos as alterações do `s3a`, registramos uma view temporária e executamos o `MERGE INTO` com `spark.sql`: o `id` 2 é atualizado e o `id` 4 é inserido.

In [ ]:
atualizacoes = spark.read.schema(schema).csv("s3a://notislake/landing/glue_delta_employees_updates/")
atualizacoes.createOrReplaceTempView("alteracoes_v")

_ = spark.sql("""
    MERGE INTO bronze.glue_delta_employees AS t
    USING alteracoes_v AS s
    ON t.id = s.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

## 6. Consultar o resultado

O estado final tem 4 linhas, o mesmo a cada reexecução. Em PySpark:

In [ ]:
echo(spark.table("bronze.glue_delta_employees").orderBy("id"), n=20)

E em Spark SQL:

In [ ]:
SELECT id, nome, area, ts FROM bronze.glue_delta_employees ORDER BY id